In [ ]:
# =========================================================
# CONFIGURAÇÃO LOCAL
# REMOVER ANTES DE SUBIR PARA O GITHUB
# =========================================================

import os

JAVA_HOME = (
    r"C:\Users\GCarapinadelima\Downloads"
    r"\microsoft-jdk-17.0.20.1-windows-x64"
    r"\jdk-17.0.20.1+1"
)

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = JAVA_HOME + r"\bin;" + os.environ["PATH"]


# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS
# ---------------------------------------------------------------------

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


# ---------------------------------------------------------------------
# CAMINHOS
# ---------------------------------------------------------------------

PROJECT_ROOT = Path(__file__).resolve().parents[3]

OUTPUT_CSV = PROJECT_ROOT / "scripts" / "Analytics" / "outputs" / "gold_05"
OUTPUT_GRAFICOS = OUTPUT_CSV / "graficos"

OUTPUT_GRAFICOS.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# FUNÇÃO PARA LER CSV
# ---------------------------------------------------------------------

def ler_csv(nome_arquivo):
    caminho = OUTPUT_CSV / nome_arquivo

    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}"
        )

    return pd.read_csv(
        caminho,
        sep=None,
        engine="python",
        encoding="utf-8-sig"
    )


# ---------------------------------------------------------------------
# FUNÇÃO PARA SALVAR GRÁFICO
# ---------------------------------------------------------------------

def salvar_grafico(nome_arquivo):
    caminho = OUTPUT_GRAFICOS / nome_arquivo

    plt.savefig(
        caminho,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    print(f"Gráfico salvo: {caminho}")


# ---------------------------------------------------------------------
# CONFIGURAÇÃO VISUAL
# ---------------------------------------------------------------------

plt.rcParams.update(
    {
        "font.size": 13,
        "axes.titlesize": 13,
        "axes.labelsize": 13,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 11
    }
)


# ---------------------------------------------------------------------
# CARREGAR OUTPUTS
# ---------------------------------------------------------------------

df_adocao = ler_csv("ia_adocao_pessoal_historico.csv")
df_prioridade = ler_csv("ia_prioridade_empresa.csv")
df_prioridade_alta = ler_csv("ia_prioridade_alta.csv")
df_uso_empresa = ler_csv("ia_uso_empresa_historico.csv")
df_uso_pessoal = ler_csv("ia_uso_pessoal_historico.csv")
df_barreiras = ler_csv("ia_barreiras_historico.csv")


# ---------------------------------------------------------------------
# ORDEM DAS EDIÇÕES
# ---------------------------------------------------------------------

EDICOES = [
    "2023-2024",
    "2024-2025",
    "2025-2026"
]


# ---------------------------------------------------------------------
# EVOLUÇÃO DA ADOÇÃO PESSOAL DE IA
# ---------------------------------------------------------------------

df_grafico_01 = df_adocao.copy()

df_grafico_01["edicao"] = pd.Categorical(
    df_grafico_01["edicao"],
    categories=EDICOES,
    ordered=True
)

df_grafico_01 = (
    df_grafico_01
    .sort_values("edicao")
)

fig, ax = plt.subplots(figsize=(14, 8))

ax.plot(
    df_grafico_01["edicao"].astype(str),
    df_grafico_01["pct_adocao_pessoal"],
    marker="o",
    linewidth=3,
    markersize=10
)

for x, y in zip(
    df_grafico_01["edicao"].astype(str),
    df_grafico_01["pct_adocao_pessoal"]
):
    ax.text(
        x,
        y + 2.5,
        f"{y:.1f}%",
        ha="center",
        va="bottom",
        fontsize=15,
        fontweight="bold"
    )

ax.set_ylim(0, 105)
ax.set_ylabel("Respondentes (%)")
ax.set_xlabel("")

ax.grid(
    axis="y",
    alpha=0.20
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.suptitle(
    "Evolução da adoção pessoal de Inteligência Artificial",
    x=0.08,
    y=0.98,
    ha="left",
    fontsize=20,
    fontweight="bold"
)

ax.set_title(
    "Percentual de respondentes que utilizam soluções de IA generativa por edição",
    loc="left",
    fontsize=13,
    pad=18
)

fig.tight_layout(rect=[0, 0, 1, 0.91])

salvar_grafico(
    "01_evolucao_adocao_pessoal_ia.png"
)

plt.show()


# ---------------------------------------------------------------------
# IA COMO PRIORIDADE ALTA NAS EMPRESAS
# ---------------------------------------------------------------------

df_grafico_02 = (
    df_prioridade_alta
    .copy()
    .sort_values("edicao")
)

fig, ax = plt.subplots(figsize=(14, 8))

barras = ax.bar(
    df_grafico_02["edicao"],
    df_grafico_02["pct_prioridade_alta"],
    width=0.55
)

for barra, valor in zip(
    barras,
    df_grafico_02["pct_prioridade_alta"]
):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 2,
        f"{valor:.1f}%",
        ha="center",
        va="bottom",
        fontsize=15,
        fontweight="bold"
    )

ax.set_ylim(0, 100)
ax.set_ylabel("Respondentes (%)")
ax.set_xlabel("")

ax.grid(
    axis="y",
    alpha=0.20
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.suptitle(
    "Inteligência Artificial como prioridade estratégica nas empresas",
    x=0.08,
    y=0.98,
    ha="left",
    fontsize=20,
    fontweight="bold"
)

ax.set_title(
    "Percentual de respondentes que classificam IA como principal prioridade ou uma das principais prioridades",
    loc="left",
    fontsize=13,
    pad=18
)

fig.tight_layout(rect=[0, 0, 1, 0.91])

salvar_grafico(
    "02_ia_prioridade_alta_empresa.png"
)

plt.show()


# ---------------------------------------------------------------------
# EVOLUÇÃO DO USO PESSOAL DE IA
# ---------------------------------------------------------------------

df_grafico_03 = df_uso_pessoal.copy()

tipos_uso = [
    "Empresa paga pela solução",
    "Uso soluções gratuitas",
    "Uso soluções tipo Copilot",
    "Uso e pago pela solução",
    "Não uso IA generativa"
]

df_grafico_03 = (
    df_grafico_03[
        df_grafico_03["tipo_uso"].isin(tipos_uso)
    ]
    .copy()
)

df_grafico_03["tipo_uso"] = pd.Categorical(
    df_grafico_03["tipo_uso"],
    categories=tipos_uso,
    ordered=True
)

df_grafico_03 = (
    df_grafico_03
    .sort_values("tipo_uso")
)

fig, ax = plt.subplots(figsize=(15, 9))

y = np.arange(len(df_grafico_03))
altura = 0.22

for i, edicao in enumerate(EDICOES):
    valores = pd.to_numeric(
        df_grafico_03[edicao],
        errors="coerce"
    )

    barras = ax.barh(
        y + ((i - 1) * altura),
        valores,
        height=altura,
        label=edicao
    )

    for barra, valor in zip(
        barras,
        valores
    ):
        if pd.notna(valor):
            ax.text(
                valor + 0.8,
                barra.get_y() + barra.get_height() / 2,
                f"{valor:.1f}%",
                va="center",
                fontsize=11
            )

ax.set_yticks(y)

ax.set_yticklabels(
    df_grafico_03["tipo_uso"]
)

ax.invert_yaxis()

ax.set_xlabel("Respondentes (%)")
ax.set_ylabel("")
ax.set_xlim(left=0)

ax.grid(
    axis="x",
    alpha=0.20
)

ax.legend(
    frameon=False,
    ncol=3,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.18)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.suptitle(
    "Evolução das formas de uso pessoal de Inteligência Artificial",
    x=0.06,
    y=0.98,
    ha="left",
    fontsize=20,
    fontweight="bold"
)

ax.set_title(
    "Formas declaradas de acesso e utilização de soluções de IA generativa por edição",
    loc="left",
    fontsize=13,
    pad=18
)

fig.tight_layout(rect=[0, 0.06, 1, 0.91])

salvar_grafico(
    "03_evolucao_uso_pessoal_ia.png"
)

plt.show()


# ---------------------------------------------------------------------
# EVOLUÇÃO DAS FORMAS DE USO DE IA NAS EMPRESAS
# ---------------------------------------------------------------------

df_grafico_04 = df_uso_empresa.copy()

tipos_empresa = [
    "Direcionamento centralizado",
    "Desenvolvedores usando Copilots",
    "Melhoria de produtos internos",
    "Melhoria de produtos externos",
    "IA como principal frente do negócio"
]

df_grafico_04 = (
    df_grafico_04[
        df_grafico_04["tipo_uso"].isin(tipos_empresa)
    ]
    .copy()
)

df_grafico_04["tipo_uso"] = pd.Categorical(
    df_grafico_04["tipo_uso"],
    categories=tipos_empresa,
    ordered=True
)

df_grafico_04 = (
    df_grafico_04
    .sort_values("tipo_uso")
)

fig, ax = plt.subplots(figsize=(15, 9))

y = np.arange(len(df_grafico_04))
altura = 0.22

for i, edicao in enumerate(EDICOES):
    valores = pd.to_numeric(
        df_grafico_04[edicao],
        errors="coerce"
    )

    barras = ax.barh(
        y + ((i - 1) * altura),
        valores,
        height=altura,
        label=edicao
    )

    for barra, valor in zip(
        barras,
        valores
    ):
        if pd.notna(valor):
            ax.text(
                valor + 0.8,
                barra.get_y() + barra.get_height() / 2,
                f"{valor:.1f}%",
                va="center",
                fontsize=11
            )

ax.set_yticks(y)

ax.set_yticklabels(
    df_grafico_04["tipo_uso"]
)

ax.invert_yaxis()

ax.set_xlabel("Respondentes (%)")
ax.set_ylabel("")
ax.set_xlim(left=0)

ax.grid(
    axis="x",
    alpha=0.20
)

ax.legend(
    frameon=False,
    ncol=3,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.18)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.suptitle(
    "Evolução das formas de uso de Inteligência Artificial nas empresas",
    x=0.06,
    y=0.98,
    ha="left",
    fontsize=20,
    fontweight="bold"
)

ax.set_title(
    "Percentual de respondentes por tipo de aplicação de IA generativa nas empresas e edição",
    loc="left",
    fontsize=13,
    pad=18
)

fig.tight_layout(rect=[0, 0.06, 1, 0.91])

salvar_grafico(
    "04_evolucao_uso_ia_empresas.png"
)

plt.show()


# ---------------------------------------------------------------------
# EVOLUÇÃO DAS PRINCIPAIS BARREIRAS PARA ADOÇÃO DE IA
# ---------------------------------------------------------------------

df_grafico_05 = df_barreiras.copy()

top_5 = (
    df_grafico_05
    .sort_values(
        "2025-2026",
        ascending=False
    )
    .head(5)["barreira"]
    .tolist()
)

df_grafico_05 = (
    df_grafico_05[
        df_grafico_05["barreira"].isin(top_5)
    ]
    .copy()
)

df_grafico_05 = (
    df_grafico_05
    .sort_values(
        "2025-2026",
        ascending=False
    )
)

fig, ax = plt.subplots(figsize=(15, 9))

y = np.arange(len(df_grafico_05))
altura = 0.22

for i, edicao in enumerate(EDICOES):
    valores = pd.to_numeric(
        df_grafico_05[edicao],
        errors="coerce"
    )

    barras = ax.barh(
        y + ((i - 1) * altura),
        valores,
        height=altura,
        label=edicao
    )

    for barra, valor in zip(
        barras,
        valores
    ):
        if pd.notna(valor):
            ax.text(
                valor + 0.7,
                barra.get_y() + barra.get_height() / 2,
                f"{valor:.1f}%",
                va="center",
                fontsize=11
            )

ax.set_yticks(y)

ax.set_yticklabels(
    df_grafico_05["barreira"]
)

ax.invert_yaxis()

ax.set_xlabel("Respondentes (%)")
ax.set_ylabel("")
ax.set_xlim(left=0)

ax.grid(
    axis="x",
    alpha=0.20
)

ax.legend(
    frameon=False,
    ncol=3,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.18)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.suptitle(
    "Evolução das principais barreiras para adoção de Inteligência Artificial",
    x=0.06,
    y=0.98,
    ha="left",
    fontsize=20,
    fontweight="bold"
)

ax.set_title(
    "Comparação histórica das cinco barreiras mais citadas na edição 2025–2026",
    loc="left",
    fontsize=13,
    pad=18
)

fig.tight_layout(rect=[0, 0.06, 1, 0.91])

salvar_grafico(
    "05_evolucao_principais_barreiras_ia.png"
)

plt.show()


# ---------------------------------------------------------------------
# FINALIZAÇÃO
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("GRÁFICOS GOLD 05 GERADOS COM SUCESSO")
print("=" * 100)

print("\nArquivos salvos em:")
print(OUTPUT_GRAFICOS)